In [2]:
import pandas as pd

X_train_full = pd.read_csv(
    "/content/X_train_full.csv"
)

y_train_full = pd.read_csv(
    "/content/y_train_full.csv"
).squeeze()

X_test = pd.read_csv(
    "/content/X_test.csv"
)

y_test = pd.read_csv(
    "/content/y_test.csv"
).squeeze()

In [3]:
split_info = {
    "random_state": 42,
    "test_size": 0.15,
    "stratified": True,
    "train_rows": len(X_train_full),
    "test_rows": len(X_test)
}

pd.Series(split_info).to_csv(
    "split_metadata.csv"
)

In [4]:
X_train_full.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous',
       'poutcome', 'age_bucket', 'contacted_before', 'prev_success',
       'balance_segment', 'campaign_bucket', 'balance_log', 'month_num',
       'month_sin', 'month_cos'],
      dtype='object')

In [8]:
!pip install catboost
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
import re # Import regular expression module

# --- Preprocessing ---
# Identify categorical columns
categorical_cols = X_train_full.select_dtypes(include='object').columns

# Apply one-hot encoding
X_train_processed = pd.get_dummies(X_train_full, columns=categorical_cols, drop_first=True)
X_test_processed = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent feature sets
train_cols = X_train_processed.columns
test_cols = X_test_processed.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test_processed[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X_train_processed[c] = 0

X_test_processed = X_test_processed[train_cols] # Ensure order and presence of columns

# Sanitize column names for XGBoost compatibility
def sanitize_column_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        new_col = re.sub(r'[^a-zA-Z0-9_]', '_', col) # Replace problematic characters with underscore
        new_cols.append(new_col)
    df.columns = new_cols
    return df

X_train_processed = sanitize_column_names(X_train_processed)
X_test_processed = sanitize_column_names(X_test_processed)

# --- Model Definitions ---
models = {
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear', max_iter=200),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0, iterations=100),
    "LightGBM": LGBMClassifier(random_state=42, n_estimators=100)
}

# --- 5-fold Stratified Cross-Validation ---
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_results = {}

print("Performing 5-fold Stratified Cross-Validation...")

for name, model in models.items():
    print(f"\n--- Model: {name} ---")
    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    roc_auc_scores = []
    pr_auc_scores = []

    for fold, (train_index, val_index) in enumerate(skf.split(X_train_processed, y_train_full)):
        X_train_fold, X_val_fold = X_train_processed.iloc[train_index], X_train_processed.iloc[val_index]
        y_train_fold, y_val_fold = y_train_full.iloc[train_index], y_train_full.iloc[val_index]

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        # For ROC-AUC and PR-AUC, we need probability predictions
        y_pred_proba = model.predict_proba(X_val_fold)[:, 1] if hasattr(model, "predict_proba") else y_pred

        accuracy_scores.append(accuracy_score(y_val_fold, y_pred))
        precision_scores.append(precision_score(y_val_fold, y_pred, zero_division=0))
        recall_scores.append(recall_score(y_val_fold, y_pred, zero_division=0))
        f1_scores.append(f1_score(y_val_fold, y_pred, zero_division=0))
        if hasattr(model, "predict_proba"):
            roc_auc_scores.append(roc_auc_score(y_val_fold, y_pred_proba))
            pr_auc_scores.append(average_precision_score(y_val_fold, y_pred_proba))
        else:
            # If model doesn't have predict_proba, use predicted classes for a 'dummy' score or skip
            roc_auc_scores.append(np.nan)
            pr_auc_scores.append(np.nan)

        print(f"Fold {fold+1}: Accuracy = {accuracy_scores[-1]:.4f}, Precision = {precision_scores[-1]:.4f}, Recall = {recall_scores[-1]:.4f}, F1 = {f1_scores[-1]:.4f}, ROC-AUC = {roc_auc_scores[-1]:.4f}, PR-AUC = {pr_auc_scores[-1]:.4f}")

    cv_results[name] = {
        "accuracy": np.mean(accuracy_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores),
        "roc_auc": np.nanmean(roc_auc_scores) if roc_auc_scores and not all(np.isnan(roc_auc_scores)) else np.nan,
        "pr_auc": np.nanmean(pr_auc_scores) if pr_auc_scores and not all(np.isnan(pr_auc_scores)) else np.nan
    }
    print(f"\nAverage CV Results for {name}:")
    print(f"  Accuracy: {cv_results[name]['accuracy']:.4f}")
    print(f"  Precision: {cv_results[name]['precision']:.4f}")
    print(f"  Recall: {cv_results[name]['recall']:.4f}")
    print(f"  F1-Score: {cv_results[name]['f1']:.4f}")
    print(f"  ROC-AUC: {cv_results[name]['roc_auc']:.4f}")
    print(f"  PR-AUC: {cv_results[name]['pr_auc']:.4f}")

# --- Final Training and Test Set Evaluation ---
print("\n--- Final Model Training and Test Set Evaluation ---")
test_results = {}

for name, model in models.items():
    print(f"\n--- Model: {name} (Final Evaluation) ---")
    # Train model on the full training data
    model.fit(X_train_processed, y_train_full)

    # Make predictions on the test set
    y_pred_test = model.predict(X_test_processed)
    y_pred_proba_test = model.predict_proba(X_test_processed)[:, 1] if hasattr(model, "predict_proba") else y_pred_test

    # Evaluate performance
    print("Classification Report:")
    print(classification_report(y_test, y_pred_test, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_test))

    test_results[name] = {
        "accuracy": accuracy_score(y_test, y_pred_test),
        "precision": precision_score(y_test, y_pred_test, zero_division=0),
        "recall": recall_score(y_test, y_pred_test, zero_division=0),
        "f1": f1_score(y_test, y_pred_test, zero_division=0)
    }
    if hasattr(model, "predict_proba"):
        test_results[name]["roc_auc"] = roc_auc_score(y_test, y_pred_proba_test)
        test_results[name]["pr_auc"] = average_precision_score(y_test, y_pred_proba_test)
    else:
        test_results[name]["roc_auc"] = np.nan
        test_results[name]["pr_auc"] = np.nan


print("\n--- Summary of Test Set Results ---")
for name, metrics in test_results.items():
    print(f"\nModel: {name}")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1']:.4f}")
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  PR-AUC: {metrics['pr_auc']:.4f}")


Performing 5-fold Stratified Cross-Validation...

--- Model: Logistic Regression ---
Fold 1: Accuracy = 0.8924, Precision = 0.6343, Recall = 0.1891, F1 = 0.2913, ROC-AUC = 0.7762, PR-AUC = 0.4092
Fold 2: Accuracy = 0.8901, Precision = 0.6015, Recall = 0.1813, F1 = 0.2786, ROC-AUC = 0.7731, PR-AUC = 0.3988
Fold 3: Accuracy = 0.8938, Precision = 0.6477, Recall = 0.2024, F1 = 0.3085, ROC-AUC = 0.7654, PR-AUC = 0.4136
Fold 4: Accuracy = 0.8909, Precision = 0.6245, Recall = 0.1702, F1 = 0.2675, ROC-AUC = 0.7748, PR-AUC = 0.4176
Fold 5: Accuracy = 0.8943, Precision = 0.6705, Recall = 0.1922, F1 = 0.2988, ROC-AUC = 0.7712, PR-AUC = 0.4260

Average CV Results for Logistic Regression:
  Accuracy: 0.8923
  Precision: 0.6357
  Recall: 0.1871
  F1-Score: 0.2889
  ROC-AUC: 0.7721
  PR-AUC: 0.4131

--- Model: Random Forest ---
Fold 1: Accuracy = 0.8929, Precision = 0.6080, Recall = 0.2380, F1 = 0.3421, ROC-AUC = 0.7761, PR-AUC = 0.4142
Fold 2: Accuracy = 0.8899, Precision = 0.5773, Recall = 0.2202, 

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:30:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 1: Accuracy = 0.8934, Precision = 0.5913, Recall = 0.2881, F1 = 0.3874, ROC-AUC = 0.7858, PR-AUC = 0.4221


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:30:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 2: Accuracy = 0.8888, Precision = 0.5553, Recall = 0.2514, F1 = 0.3461, ROC-AUC = 0.7877, PR-AUC = 0.4226


...

Fold 4: Accuracy = 0.8931, Precision = 0.5970, Recall = 0.2670, F1 = 0.3689, ROC-AUC = 0.7880, PR-AUC = 0.4389


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:30:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 5: Accuracy = 0.8980, Precision = 0.6487, Recall = 0.2811, F1 = 0.3922, ROC-AUC = 0.7801, PR-AUC = 0.4530

Average CV Results for XGBoost:
  Accuracy: 0.8928
  Precision: 0.5917
  Recall: 0.2727
  F1-Score: 0.3731
  ROC-AUC: 0.7861
  PR-AUC: 0.4343

--- Model: CatBoost ---
Fold 1: Accuracy = 0.8918, Precision = 0.5821, Recall = 0.2681, F1 = 0.3671, ROC-AUC = 0.7884, PR-AUC = 0.4319
Fold 2: Accuracy = 0.8926, Precision = 0.5954, Recall = 0.2570, F1 = 0.3590, ROC-AUC = 0.7957, PR-AUC = 0.4362
Fold 3: Accuracy = 0.8952, Precision = 0.6205, Recall = 0.2692, F1 = 0.3755, ROC-AUC = 0.7939, PR-AUC = 0.4463
Fold 4: Accuracy = 0.8938, Precision = 0.6045, Recall = 0.2670, F1 = 0.3704, ROC-AUC = 0.7953, PR-AUC = 0.4286
Fold 5: Accuracy = 0.8969, Precision = 0.6436, Recall = 0.2689, F1 = 0.3793, ROC-AUC = 0.7884, PR-AUC = 0.4608

Average CV Results for CatBoost:
  Accuracy: 0.8941
  Precision: 0.6092
  Recall: 0.2660
  F1-Score: 0.3702
  ROC-AUC: 0.7924
  PR-AUC: 0.4408

--- Model: LightGBM -

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:30:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.98      0.94      5987
           1       0.56      0.24      0.34       793

    accuracy                           0.89      6780
   macro avg       0.73      0.61      0.64      6780
weighted avg       0.87      0.89      0.87      6780

Confusion Matrix:
[[5839  148]
 [ 602  191]]

--- Model: CatBoost (Final Evaluation) ---
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.98      0.94      5987
           1       0.56      0.24      0.34       793

    accuracy                           0.89      6780
   macro avg       0.74      0.61      0.64      6780
weighted avg       0.87      0.89      0.87      6780

Confusion Matrix:
[[5840  147]
 [ 603  190]]

--- Model: LightGBM (Final Evaluation) ---
[LightGBM] [Info] Number of positive: 4496, number of negative: 33919
[LightGBM] [Info] Auto-choosing row-wise multi-threa

Next: Calibration, Incremental Lift, Campaign ROI, Conversion Rate in Top-K, Decile Analysis, Lift@10%

Imbalance Handling, Hyperparameter tunning